W2D2 — Lab 1: Give the Model an API

In [1]:
!pip -q install \
    "fastapi==0.115.*" \
    "uvicorn[standard]==0.32.*" \
    "pydantic==2.9.*" \
    "transformers==4.46.*" \
    "accelerate==1.1.*" \
    "openai==1.54.*" \
    "httpx==0.27.*"

In [2]:
%%writefile schemas.py

from __future__ import annotations

from typing import List, Literal, Optional, Union
from pydantic import BaseModel, Field


class ChatMessage(BaseModel):
    role: Literal["system", "user", "assistant"]
    content: str


class ChatCompletionRequest(BaseModel):
    model: str
    messages: List[ChatMessage]
    max_tokens: int = Field(default=256, ge=1)
    temperature: float = Field(default=0.7, ge=0.0, le=2.0)
    stream: bool = False
    tools: Optional[List[dict]] = None
    tool_choice: Optional[Union[str, dict]] = None


class ResponseMessage(BaseModel):
    role: Literal["assistant"] = "assistant"
    content: str


class Choice(BaseModel):
    index: int = 0
    message: ResponseMessage
    finish_reason: Literal["stop", "length"] = "stop"


class Usage(BaseModel):
    prompt_tokens: int = 0
    completion_tokens: int = 0
    total_tokens: int = 0


class ChatCompletionResponse(BaseModel):
    id: str
    object: Literal["chat.completion"] = "chat.completion"
    created: int
    model: str
    choices: List[Choice]
    usage: Usage


class ModelCard(BaseModel):
    id: str
    object: Literal["model"] = "model"
    created: int
    owned_by: str = "aidc"


class ModelList(BaseModel):
    object: Literal["list"] = "list"
    data: List[ModelCard]


class HealthResponse(BaseModel):
    status: Literal["ok"] = "ok"
    model: str

Overwriting schemas.py


In [3]:
%%writefile main.py

from __future__ import annotations

import json
import os
import time
import uuid

import torch
from fastapi import FastAPI, HTTPException
from fastapi.responses import StreamingResponse
from transformers import AutoModelForCausalLM, AutoTokenizer

from schemas import (
    ChatCompletionRequest,
    ChatCompletionResponse,
    Choice,
    HealthResponse,
    ModelCard,
    ModelList,
    ResponseMessage,
    Usage,
)

MODEL_ID = os.environ.get(
    "MODEL_ID",
    "Qwen/Qwen2.5-0.5B-Instruct"
)

app = FastAPI(
    title="serving-stack",
    version="wk2"
)

print(f"Loading {MODEL_ID} on CPU...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float32
)

model.to("cpu")
model.eval()

print("Model ready")


@app.get("/health", response_model=HealthResponse)
def health() -> HealthResponse:
    return HealthResponse(
        status="ok",
        model=MODEL_ID
    )


@app.get("/v1/models", response_model=ModelList)
def list_models() -> ModelList:
    return ModelList(
        data=[
            ModelCard(
                id=MODEL_ID,
                created=int(time.time()),
                owned_by="aidc"
            )
        ]
    )


def _build_inputs(req: ChatCompletionRequest):
    input_ids = tokenizer.apply_chat_template(
        [m.model_dump() for m in req.messages],
        add_generation_prompt=True,
        return_tensors="pt",
    )
    return input_ids, input_ids.shape[1]


def _generate(
    input_ids,
    req: ChatCompletionRequest
):
    with torch.no_grad():
        out = model.generate(
            input_ids,
            attention_mask=torch.ones_like(input_ids),
            max_new_tokens=req.max_tokens,
            do_sample=req.temperature > 0,
            temperature=(
                req.temperature
                if req.temperature > 0
                else None
            ),
            pad_token_id=tokenizer.eos_token_id,
        )

    return out[0][input_ids.shape[1]:]


@app.post(
    "/v1/chat/completions",
    response_model=None
)
def chat_completions(
    req: ChatCompletionRequest
):
    if req.model != MODEL_ID:
        raise HTTPException(
            status_code=400,
            detail={
                "error": {
                    "message": (
                        f"model '{req.model}' not found"
                    ),
                    "type": "invalid_request_error",
                    "code": "model_not_found"
                }
            },
        )

    input_ids, prompt_tokens = _build_inputs(req)

    if req.stream:
        return _stream(
            input_ids,
            prompt_tokens,
            req
        )

    new_tokens = _generate(
        input_ids,
        req
    )

    completion_tokens = int(
        new_tokens.shape[0]
    )

    text = tokenizer.decode(
        new_tokens,
        skip_special_tokens=True
    )

    return ChatCompletionResponse(
        id="chatcmpl-" + uuid.uuid4().hex,
        created=int(time.time()),
        model=req.model,
        choices=[
            Choice(
                index=0,
                message=ResponseMessage(
                    role="assistant",
                    content=text
                ),
                finish_reason=(
                    "length"
                    if completion_tokens >= req.max_tokens
                    else "stop"
                ),
            )
        ],
        usage=Usage(
            prompt_tokens=prompt_tokens,
            completion_tokens=completion_tokens,
            total_tokens=(
                prompt_tokens + completion_tokens
            ),
        ),
    )


def _stream(
    input_ids,
    prompt_tokens: int,
    req: ChatCompletionRequest
):
    new_tokens = _generate(
        input_ids,
        req
    )

    cid = "chatcmpl-" + uuid.uuid4().hex
    created = int(time.time())

    def chunk(delta: dict, finish=None):
        payload = {
            "id": cid,
            "object": "chat.completion.chunk",
            "created": created,
            "model": req.model,
            "choices": [
                {
                    "index": 0,
                    "delta": delta,
                    "finish_reason": finish
                }
            ],
        }

        return (
            "data: "
            + json.dumps(payload)
            + "\n\n"
        )

    def events():
        yield chunk(
            {
                "role": "assistant",
                "content": ""
            }
        )

        for tok in new_tokens:
            piece = tokenizer.decode(
                [tok],
                skip_special_tokens=True
            )

            if piece:
                yield chunk(
                    {"content": piece}
                )

        yield chunk(
            {},
            finish="stop"
        )

        yield "data: [DONE]\n\n"

    return StreamingResponse(
        events(),
        media_type="text/event-stream"
    )

Overwriting main.py


In [4]:
import subprocess
import time

server = subprocess.Popen(
    [
        "uvicorn",
        "main:app",
        "--host",
        "0.0.0.0",
        "--port",
        "8000"
    ]
)

time.sleep(8)

print("Server process:", server.pid)

Server process: 15749


In [6]:
import requests

r = requests.get(
    "http://localhost:8000/health"
)

print(r.status_code)
print(r.json())

200
{'status': 'ok', 'model': 'Qwen/Qwen2.5-0.5B-Instruct'}


In [7]:
r = requests.get(
    "http://localhost:8000/v1/models"
)

print(r.status_code)
print(r.json())

200
{'object': 'list', 'data': [{'id': 'Qwen/Qwen2.5-0.5B-Instruct', 'object': 'model', 'created': 1787591671, 'owned_by': 'aidc'}]}


In [8]:
payload = {
    "model": "Qwen/Qwen2.5-0.5B-Instruct",
    "messages": [
        {
            "role": "user",
            "content": "Say hello in one word."
        }
    ],
    "max_tokens": 16
}

r = requests.post(
    "http://localhost:8000/v1/chat/completions",
    json=payload
)

print("Status:", r.status_code)
print(r.json())

Status: 200
{'id': 'chatcmpl-6b1818a4aa734cf089f2b653f310b11c', 'object': 'chat.completion', 'created': 1787591677, 'model': 'Qwen/Qwen2.5-0.5B-Instruct', 'choices': [{'index': 0, 'message': {'role': 'assistant', 'content': 'Hello!'}, 'finish_reason': 'stop'}], 'usage': {'prompt_tokens': 35, 'completion_tokens': 3, 'total_tokens': 38}}


In [9]:
from openai import OpenAI

client = OpenAI(
    base_url="http://localhost:8000/v1",
    api_key="not-needed"
)

resp = client.chat.completions.create(
    model="Qwen/Qwen2.5-0.5B-Instruct",
    messages=[
        {
            "role": "system",
            "content": "You are a terse assistant."
        },
        {
            "role": "user",
            "content": "Name three primary colours."
        }
    ],
    max_tokens=64
)

print("Reply:", resp.choices[0].message.content)
print("Finish reason:", resp.choices[0].finish_reason)
print("Usage:", resp.usage)

Reply: Three primary colors are red, yellow, and blue.
Finish reason: stop
Usage: CompletionUsage(completion_tokens=12, prompt_tokens=24, total_tokens=36, completion_tokens_details=None, prompt_tokens_details=None)


W2D2 — Lab 2

In [10]:
%%writefile schemas.py

from __future__ import annotations

from typing import List, Literal, Optional, Union
from pydantic import (
    BaseModel,
    Field,
    field_validator
)

class ChatMessage(BaseModel):
    role: Literal[
        "system",
        "user",
        "assistant"
    ]
    content: str

class ChatCompletionRequest(BaseModel):
    model: str
    messages: List[ChatMessage] = Field(
        ...,
        min_length=1
    )
    max_tokens: int = Field(
        default=256,
        ge=1
    )
    temperature: float = Field(
        default=0.7,
        ge=0.0,
        le=2.0
    )
    stream: bool = False
    tools: Optional[List[dict]] = None
    tool_choice: Optional[
        Union[str, dict]
    ] = None

    @field_validator("messages")
    @classmethod
    def last_message_must_be_user_or_system(
        cls,
        value
    ):
        if (
            value
            and value[-1].role == "assistant"
        ):
            raise ValueError(
                "the last message must be from "
                "'user' or 'system', not 'assistant'"
            )
        return value

class ResponseMessage(BaseModel):
    role: Literal["assistant"] = "assistant"
    content: str

class Choice(BaseModel):
    index: int = 0
    message: ResponseMessage
    finish_reason: Literal[
        "stop",
        "length"
    ] = "stop"

class Usage(BaseModel):
    prompt_tokens: int = 0
    completion_tokens: int = 0
    total_tokens: int = 0

class ChatCompletionResponse(BaseModel):
    id: str
    object: Literal[
        "chat.completion"
    ] = "chat.completion"
    created: int
    model: str
    choices: List[Choice]
    usage: Usage

class ModelCard(BaseModel):
    id: str
    object: Literal["model"] = "model"
    created: int
    owned_by: str = "aidc"

class ModelList(BaseModel):
    object: Literal["list"] = "list"
    data: List[ModelCard]

class HealthResponse(BaseModel):
    status: Literal["ok"] = "ok"
    model: str

Overwriting schemas.py


In [11]:
server.terminate()
server.wait()

print("Old server stopped")

Old server stopped


In [12]:
import subprocess
import time

server = subprocess.Popen(
    [
        "uvicorn",
        "main:app",
        "--host",
        "0.0.0.0",
        "--port",
        "8000"
    ]
)

time.sleep(8)

print("Server restarted")

Server restarted


In [13]:
%%writefile fuzz_client.py
import os
import sys
import time
import concurrent.futures

import httpx


BASE_URL = os.environ.get(
    "BASE_URL",
    "http://localhost:8000"
)

MODEL = os.environ.get(
    "MODEL_ID",
    "Qwen/Qwen2.5-0.5B-Instruct"
)

results = []


def case(
    name,
    expected_status,
    payload=None,
    raw_body=None,
    headers=None
):
    expected = (
        {expected_status}
        if isinstance(expected_status, int)
        else set(expected_status)
    )

    hdrs = {
        "Content-Type": "application/json"
    }

    if headers:
        hdrs.update(headers)

    try:
        if raw_body is not None:
            r = httpx.post(
                f"{BASE_URL}/v1/chat/completions",
                content=raw_body,
                headers=hdrs,
                timeout=30
            )
        else:
            r = httpx.post(
                f"{BASE_URL}/v1/chat/completions",
                json=payload,
                headers=hdrs,
                timeout=30
            )

        ok = r.status_code in expected

        results.append(
            (
                name,
                ok,
                r.status_code,
                expected
            )
        )

        return ok, r

    except Exception as exc:
        results.append(
            (
                name,
                False,
                f"EXCEPTION: {exc}",
                expected
            )
        )

        return False, None


def run_cases():

    # Invalid requests: expected 422

    case(
        "missing messages",
        422,
        payload={
            "model": MODEL,
            "max_tokens": 16
        }
    )

    case(
        "missing model",
        422,
        payload={
            "messages": [
                {
                    "role": "user",
                    "content": "hi"
                }
            ]
        }
    )

    case(
        "messages is string",
        422,
        payload={
            "model": MODEL,
            "messages": "hi"
        }
    )

    case(
        "empty messages",
        422,
        payload={
            "model": MODEL,
            "messages": []
        }
    )

    case(
        "invalid role",
        422,
        payload={
            "model": MODEL,
            "messages": [
                {
                    "role": "wizard",
                    "content": "hi"
                }
            ]
        }
    )

    case(
        "negative max_tokens",
        422,
        payload={
            "model": MODEL,
            "messages": [
                {
                    "role": "user",
                    "content": "hi"
                }
            ],
            "max_tokens": -5
        }
    )

    case(
        "max_tokens zero",
        422,
        payload={
            "model": MODEL,
            "messages": [
                {
                    "role": "user",
                    "content": "hi"
                }
            ],
            "max_tokens": 0
        }
    )

    case(
        "temperature out of range",
        422,
        payload={
            "model": MODEL,
            "messages": [
                {
                    "role": "user",
                    "content": "hi"
                }
            ],
            "temperature": 5.0
        }
    )

    case(
        "assistant is last message",
        422,
        payload={
            "model": MODEL,
            "messages": [
                {
                    "role": "assistant",
                    "content": "hi"
                }
            ]
        }
    )

    case(
        "malformed JSON",
        422,
        raw_body=b'{"model": "x", "messages": [',
        headers={
            "Content-Type": "application/json"
        }
    )

    # Unusual but valid: expected 200

    case(
        "unicode and emoji",
        200,
        payload={
            "model": MODEL,
            "messages": [
                {
                    "role": "user",
                    "content": "hello 👋 世界"
                }
            ],
            "max_tokens": 16
        }
    )

    case(
        "minimal valid request",
        200,
        payload={
            "model": MODEL,
            "messages": [
                {
                    "role": "user",
                    "content": "hi"
                }
            ],
            "max_tokens": 16
        }
    )


def run_concurrency_probe(n=2):

    payload = {
        "model": MODEL,
        "messages": [
            {
                "role": "user",
                "content": "hi"
            }
        ],
        "max_tokens": 16
    }

    def one():
        t0 = time.time()

        httpx.post(
            f"{BASE_URL}/v1/chat/completions",
            json=payload,
            timeout=30
        )

        return time.time() - t0

    wall_start = time.time()

    with concurrent.futures.ThreadPoolExecutor(
        max_workers=n
    ) as executor:
        durations = list(
            executor.map(
                lambda _: one(),
                range(n)
            )
        )

    wall = time.time() - wall_start
    serial_estimate = sum(durations)

    verdict = (
        "looks serial"
        if wall > 0.8 * serial_estimate
        else "looks concurrent"
    )

    print(
        f"\nConcurrency probe: "
        f"wall={wall:.3f}s, "
        f"sum={serial_estimate:.3f}s "
        f"({verdict})"
    )


def main():
    run_cases()

    n_pass = sum(
        1
        for _, ok, _, _ in results
        if ok
    )

    n_total = len(results)

    for name, ok, status, expected in results:
        mark = "PASS" if ok else "FAIL"

        print(
            f"[{mark}] "
            f"{name:35s} "
            f"got={status} "
            f"expected={expected}"
        )

    run_concurrency_probe()

    print(
        f"\n{n_pass}/{n_total} cases passed"
    )

    print(
        "GREEN CHECK: PASS"
        if n_pass == n_total
        else "GREEN CHECK: FAIL"
    )

    sys.exit(
        0 if n_pass == n_total else 1
    )


if __name__ == "__main__":
    main()

Overwriting fuzz_client.py


In [14]:
!python fuzz_client.py

[PASS] missing messages                    got=422 expected={422}
[PASS] missing model                       got=422 expected={422}
[PASS] messages is string                  got=422 expected={422}
[PASS] empty messages                      got=422 expected={422}
[PASS] invalid role                        got=422 expected={422}
[PASS] negative max_tokens                 got=422 expected={422}
[PASS] max_tokens zero                     got=422 expected={422}
[PASS] temperature out of range            got=422 expected={422}
[PASS] assistant is last message           got=422 expected={422}
[PASS] malformed JSON                      got=422 expected={422}
[PASS] unicode and emoji                   got=200 expected={200}
[PASS] minimal valid request               got=200 expected={200}

Concurrency probe: wall=7.762s, sum=15.503s (looks concurrent)

12/12 cases passed
GREEN CHECK: PASS


W2D2 — Lab 3

In [15]:
!pip show transformers | grep Version

Version: 4.46.3


In [23]:


messages = [
    {
        "role": "user",
        "content": "hi"
    }
]

result = main.tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    return_tensors="pt"
)

print(type(result))

<class 'torch.Tensor'>


In [31]:
encoded =main.tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    return_tensors="pt",
    return_dict=True
)

input_ids = encoded["input_ids"]

print(type(encoded))
print(input_ids.shape)

<class 'transformers.tokenization_utils_base.BatchEncoding'>
torch.Size([1, 30])


In [35]:
from openai import OpenAI

client = OpenAI(
    base_url="http://localhost:8000/v1",
    api_key="not-needed"
)

response = client.chat.completions.create(
    model="Qwen/Qwen2.5-0.5B-Instruct",
    messages=[
        {
            "role": "user",
            "content": "Say hello Ghala in two word"
        }
    ],
    max_tokens=16
)

print(response.choices[0].message.content)

Hello, Ghala!
